# Intertemporal choice: XGBoost (standalone notebook)

**Self-contained:** uses only **`clean_data_basic.csv`** — the same base file as `choice_model_readable.ipynb` and `data_cleaning.py`.

## What the model predicts

Whether the participant chooses **LL** (patient: wait for the larger later reward) or **SS** (impatient: take the smaller sooner reward), evaluated on **held-out subjects** (no trial from those people was used in training).

## Why add XGBoost

| Idea | What to say |
|------|-------------|
| **Gradient boosted trees** | Sequences of trees correct previous mistakes; often strong on tabular data with nonlinear effects. |
| **Compared to Random Forest** | RF averages many deep-ish trees; XGBoost grows an **ensemble in stages** with regularization (`subsample`, `colsample_bytree`, `max_depth`). |
| **Compared to logistic regression** | More flexible decision boundaries; **less** direct than log-odds coefficients — use importances / SHAP to explain. |
| **Same evaluation rule** | **Subject-level** train/test split so within-person dependence is respected. |

## Metrics (plain language)

- **Accuracy** — Share of test **trials** predicted correctly.
- **ROC-AUC** — Quality of **ranking** SS vs LL by predicted probability (0.5 random, 1.0 perfect).
- **Precision / recall (LL)** — Precision: when the model says LL, how often right. Recall: of true LL trials, how many detected.

**Method note:** `patience_rate` is computed only from **training subjects’ training rows** (no label leakage from the test set).

## Requirements

`pandas`, `numpy`, `scikit-learn`, `matplotlib`, **`xgboost`**. Install: `pip install xgboost`. Optional: `shap` if `RUN_SHAP = True`.

## How to use

1. Put `clean_data_basic.csv` next to this notebook (or set `CSV_PATH` to an absolute path).
2. Edit **Configuration**, then **Run all**.

## macOS: `Library not loaded: libomp.dylib`

The `xgboost` pip wheel links to **OpenMP** (`libomp`). If you see `XGBoostError` / `libomp.dylib` missing:

1. Install OpenMP (Apple Silicon Homebrew):

   ```bash
   brew install libomp
   ```

2. **Quit and restart** Cursor (or restart the Jupyter kernel) so the loader picks up the library.

3. **Intel Mac** (Homebrew in `/usr/local`): if needed, after `brew install libomp`, the library is under `/usr/local/opt/libomp/lib/`.

If you cannot install `libomp`, set **`USE_SKLEARN_HISTGB_FALLBACK = True`** in **Configuration** below. The notebook will use **sklearn’s `HistGradientBoostingClassifier`** instead (similar boosted trees, no separate OpenMP install).

If XGBoost warns about **32-bit Python on a 64-bit Mac**, install a **64-bit** Python (e.g. `python3` from Homebrew or python.org) and recreate your venv.


In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [ ]:
CSV_PATH = Path("clean_data_basic.csv")
OUT_DIR = Path("results_xgb")  # None to skip saving files
NROWS = None  # None = full CSV; e.g. 100_000 for speed
TEST_SIZE_SUBJECTS = 0.2
SEED = 42

# XGBoost (tune gently; lower learning_rate often needs more trees)
XGB_N_ESTIMATORS = 400
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.05
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE_BYTREE = 0.8
# Rebalance LL vs SS using class counts (set False for no weighting)
USE_SCALE_POS_WEIGHT = True

RUN_LOGISTIC_BASELINE = True
RUN_SHAP = False
SHAP_SAMPLE = 2000

# True = skip XGBoost; use sklearn HistGradientBoosting (fixes missing libomp on Mac)
USE_SKLEARN_HISTGB_FALLBACK = False


## Pipeline code (load → features → split → train → evaluate)

In [ ]:
from typing import Optional, Union

BASE_READ_COLS = [
    "choice",
    "subj_ident",
    "ss_value",
    "ss_time",
    "ll_value",
    "ll_time",
    "rt",
    "trial_idx",
    "time_pressure",
    "online_study",
    "incentivization",
    "value_diff",
    "time_diff_days",
]


def load_clean_csv(path: Path, nrows: Optional[int]) -> pd.DataFrame:
    t0 = time.perf_counter()
    header = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    usecols = [c for c in BASE_READ_COLS if c in header]
    if "choice" not in usecols or "subj_ident" not in usecols:
        raise ValueError(f"CSV missing choice/subj_ident. Available: {usecols}")
    kw = dict(usecols=usecols, low_memory=False, engine="c")
    if nrows is not None:
        kw["nrows"] = nrows
    df = pd.read_csv(path, **kw)
    print(f"Loaded {len(df):,} rows × {df.shape[1]} columns in {time.perf_counter() - t0:.1f}s")
    return df


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    ss = pd.to_numeric(out["ss_value"], errors="coerce")
    ll = pd.to_numeric(out["ll_value"], errors="coerce")
    out["reward_ratio"] = np.where(ss.fillna(0) > 0, ll / ss, np.nan)
    out["delay_diff"] = pd.to_numeric(out["ll_time"], errors="coerce") - pd.to_numeric(
        out["ss_time"], errors="coerce"
    )
    if "time_diff_days" in out.columns:
        out["delay_diff"] = out["delay_diff"].fillna(pd.to_numeric(out["time_diff_days"], errors="coerce"))
    rt = pd.to_numeric(out["rt"], errors="coerce")
    out["rt_log"] = np.log(rt.where(rt > 0))
    if "incentivization" in out.columns:
        inc = out["incentivization"].astype("string")
    else:
        inc = pd.Series("__na__", index=out.index, dtype="string")
    out["incentivization_code"] = pd.factorize(inc.fillna("__na__"))[0]
    D = out["delay_diff"]
    ratio = out["reward_ratio"]
    k_thr = (ratio - 1) / D.where(D > 0)
    out["log_discount_k"] = -np.log(k_thr.where(k_thr > 0))
    return out


def add_patience_rate_no_leakage(df: pd.DataFrame, train_subjects: np.ndarray) -> pd.DataFrame:
    out = df.copy()
    tr_mask = out["subj_ident"].isin(train_subjects)
    train_rows = out.loc[tr_mask]
    subj_mean = train_rows.groupby("subj_ident", sort=False)["choice"].mean()
    global_mean = float(train_rows["choice"].mean())
    out["patience_rate"] = out["subj_ident"].map(subj_mean).fillna(global_mean)
    return out


def preferred_feature_columns(df: pd.DataFrame) -> list[str]:
    cols = [
        "ss_value",
        "ss_time",
        "ll_value",
        "ll_time",
        "rt",
        "trial_idx",
        "reward_ratio",
        "delay_diff",
        "rt_log",
        "log_discount_k",
        "patience_rate",
        "time_pressure",
        "online_study",
        "incentivization_code",
    ]
    return [c for c in cols if c in df.columns]


def run_xgboost_pipeline(
    *,
    csv_path: Path,
    out_dir: Optional[Path],
    nrows: Optional[int],
    test_size_subjects: float,
    seed: int,
    xgb_n_estimators: int,
    xgb_max_depth: int,
    xgb_learning_rate: float,
    xgb_subsample: float,
    xgb_colsample_bytree: float,
    use_scale_pos_weight: bool,
    run_logistic_baseline: bool,
    run_shap: bool,
    shap_sample: int,
    use_sklearn_histgb_fallback: bool = False,
) -> None:
    lr_pred = None
    use_histgb = bool(use_sklearn_histgb_fallback)
    boost_step = "xgb"
    model_title = "XGBoost"

    if not use_histgb:
        try:
            from xgboost import XGBClassifier
        except Exception as e:
            print("XGBoost could not be loaded:", e)
            print("→ Using sklearn HistGradientBoostingClassifier instead (no libomp).")
            print("  For real XGBoost on Mac:  brew install libomp  then restart the kernel.")
            use_histgb = True

    df = load_clean_csv(csv_path, nrows)
    df["choice"] = pd.to_numeric(df["choice"], errors="coerce")
    df = df[df["choice"].isin([0, 1])].copy()
    df["choice"] = df["choice"].astype(int)
    df["subj_ident"] = df["subj_ident"].astype(str).str.strip()

    df = add_engineered_features(df)

    subjects = df["subj_ident"].unique()
    train_subj, test_subj = train_test_split(
        subjects, test_size=test_size_subjects, random_state=seed
    )
    train_subj = np.asarray(train_subj)
    test_subj = np.asarray(test_subj)

    df = add_patience_rate_no_leakage(df, train_subj)

    feature_cols = preferred_feature_columns(df)
    need = feature_cols + ["choice"]
    before = len(df)
    df = df.dropna(subset=need)
    print(f"Rows after dropna on model features: {len(df):,} (dropped {before - len(df):,})")

    train_mask = df["subj_ident"].isin(train_subj)
    test_mask = df["subj_ident"].isin(test_subj)
    train = df.loc[train_mask]
    test = df.loc[test_mask]

    X_train = train[feature_cols]
    y_train = train["choice"]
    X_test = test[feature_cols]
    y_test = test["choice"]

    n_test_trials = len(y_test)
    print(f"\nTrain trials: {len(y_train):,} | Test trials: {n_test_trials:,}")
    print(f"Train subjects: {train['subj_ident'].nunique():,} | Test subjects: {test['subj_ident'].nunique():,}")

    spw = None
    if use_scale_pos_weight and not use_histgb:
        n0 = int((y_train == 0).sum())
        n1 = int((y_train == 1).sum())
        spw = n0 / max(n1, 1)
        print(f"scale_pos_weight (train): {spw:.4f}  (weight for class 1 / LL)")
    elif use_scale_pos_weight and use_histgb:
        print("class_weight='balanced' on HistGradientBoosting (train distribution)")

    if use_histgb:
        from sklearn.ensemble import HistGradientBoostingClassifier

        boost_step = "hgb"
        model_title = "HistGradientBoosting (sklearn fallback)"
        cw = "balanced" if use_scale_pos_weight else None
        clf = HistGradientBoostingClassifier(
            max_iter=max(100, xgb_n_estimators),
            learning_rate=xgb_learning_rate,
            max_depth=xgb_max_depth,
            random_state=seed,
            class_weight=cw,
        )
        boost_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), (boost_step, clf)])
    else:
        xgb_kw = dict(
            n_estimators=xgb_n_estimators,
            max_depth=xgb_max_depth,
            learning_rate=xgb_learning_rate,
            subsample=xgb_subsample,
            colsample_bytree=xgb_colsample_bytree,
            random_state=seed,
            n_jobs=-1,
            eval_metric="logloss",
            tree_method="hist",
        )
        if spw is not None:
            xgb_kw["scale_pos_weight"] = spw
        boost_pipe = Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("xgb", XGBClassifier(**xgb_kw)),
            ]
        )

    t1 = time.perf_counter()
    boost_pipe.fit(X_train, y_train)
    print(f"{model_title} fit time: {time.perf_counter() - t1:.1f}s")

    xgb_pred = boost_pipe.predict(X_test)
    xgb_proba = boost_pipe.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, xgb_pred)
    auc = roc_auc_score(y_test, xgb_proba)
    prec = precision_score(y_test, xgb_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, xgb_pred, pos_label=1, zero_division=0)

    print("\n" + "=" * 60)
    print(f"{model_title.upper()} — held-out subjects")
    print("=" * 60)
    print(
        f"The model answers the SS-vs-LL question {n_test_trials:,} times on test trials "
        f"(subject-level holdout)."
    )
    print(f"\nAccuracy:  {acc:.4f}  (~{acc*100:.1f}% of test trial predictions correct)")
    print(f"ROC-AUC:   {auc:.4f}  (1.0 = perfect ranking, 0.5 = random)")
    print(f"Precision (LL / 'patient'): {prec:.4f}")
    print(f"Recall (LL):                {rec:.4f}")
    print("\n" + classification_report(y_test, xgb_pred, target_names=["SS (impatient)", "LL (patient)"]))

    if run_logistic_baseline:
        imputer = SimpleImputer(strategy="median")
        X_tr_i = imputer.fit_transform(X_train)
        X_te_i = imputer.transform(X_test)
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr_i)
        X_te_sc = scaler.transform(X_te_i)
        lr = LogisticRegression(max_iter=2000, random_state=seed)
        lr.fit(X_tr_sc, y_train)
        lr_pred = lr.predict(X_te_sc)
        lr_proba = lr.predict_proba(X_te_sc)[:, 1]
        print("\n── Logistic baseline (same split, median impute + scale) ──")
        print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}  ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")

    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)

    boost_model = boost_pipe.named_steps[boost_step]
    imp_df = (
        pd.DataFrame({"feature": feature_cols, "importance": boost_model.feature_importances_})
        .sort_values("importance", ascending=True)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(imp_df["feature"], imp_df["importance"], color="darkorange", edgecolor="white")
    ax.set_xlabel("Feature importance")
    ax.set_title(f"{model_title} — feature importance (clean_data_basic.csv)", fontweight="bold")
    fig.tight_layout()
    if out_dir is not None:
        fig.savefig(out_dir / "plot_xgb_importance.png", bbox_inches="tight", dpi=150)
    plt.show()

    if run_logistic_baseline and lr_pred is not None:
        fig2, axes = plt.subplots(1, 2, figsize=(12, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, lr_pred, ax=axes[0], display_labels=["SS (impatient)", "LL (patient)"], colorbar=False
        )
        axes[0].set_title("Logistic regression", fontweight="bold")
        ConfusionMatrixDisplay.from_predictions(
            y_test, xgb_pred, ax=axes[1], display_labels=["SS (impatient)", "LL (patient)"], colorbar=False
        )
        axes[1].set_title(model_title, fontweight="bold")
    else:
        fig2, ax_cm = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, xgb_pred, ax=ax_cm, display_labels=["SS (impatient)", "LL (patient)"], colorbar=False
        )
        ax_cm.set_title(model_title, fontweight="bold")
    fig2.suptitle("Confusion matrices — held-out subjects", fontsize=13, fontweight="bold")
    fig2.tight_layout()
    if out_dir is not None:
        fig2.savefig(out_dir / "plot_confusion_matrix.png", bbox_inches="tight", dpi=150)
    plt.show()

    if run_shap:
        try:
            import shap
        except ImportError:
            print("SHAP not installed; pip install shap or set RUN_SHAP = False")
        else:
            X_samp = X_test.sample(n=min(shap_sample, len(X_test)), random_state=seed)
            try:
                explainer = shap.TreeExplainer(boost_model)
                shap_values = explainer.shap_values(X_samp)
            except Exception as e:
                print("SHAP TreeExplainer failed for this estimator:", e)
                shap_values = None
            if shap_values is not None:
                if isinstance(shap_values, list):
                    shap_ll = np.asarray(shap_values[1])
                else:
                    sv = np.asarray(shap_values)
                    shap_ll = sv[..., 1] if sv.ndim == 3 else sv
                plt.figure(figsize=(10, 6))
                shap.summary_plot(
                    shap_ll, X_samp, feature_names=feature_cols, plot_type="bar", show=False
                )
                plt.title(f"SHAP — {model_title}", fontweight="bold")
                plt.tight_layout()
                if out_dir is not None:
                    plt.savefig(out_dir / "plot_shap_bar.png", bbox_inches="tight", dpi=150)
                plt.show()

    metrics = {
        "csv": str(csv_path.resolve()),
        "backend": model_title,
        "n_test_trials": int(n_test_trials),
        "n_test_subjects": int(test["subj_ident"].nunique()),
        "accuracy": float(acc),
        "roc_auc": float(auc),
        "precision_ll": float(prec),
        "recall_ll": float(rec),
        "features": feature_cols,
    }
    if out_dir is not None:
        with open(out_dir / "xgb_metrics.json", "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)
        print(f"\nSaved metrics + plots under: {out_dir.resolve()}")


## Run

In [ ]:
run_xgboost_pipeline(
    csv_path=CSV_PATH,
    out_dir=OUT_DIR,
    nrows=NROWS,
    test_size_subjects=TEST_SIZE_SUBJECTS,
    seed=SEED,
    xgb_n_estimators=XGB_N_ESTIMATORS,
    xgb_max_depth=XGB_MAX_DEPTH,
    xgb_learning_rate=XGB_LEARNING_RATE,
    xgb_subsample=XGB_SUBSAMPLE,
    xgb_colsample_bytree=XGB_COLSAMPLE_BYTREE,
    use_scale_pos_weight=USE_SCALE_POS_WEIGHT,
    run_logistic_baseline=RUN_LOGISTIC_BASELINE,
    run_shap=RUN_SHAP,
    shap_sample=SHAP_SAMPLE,
    use_sklearn_histgb_fallback=USE_SKLEARN_HISTGB_FALLBACK,
)